## 광명시 시설물 데이터 + 구조 점수 산출값을 병합해 광명시 데이터만들기

5_gwangmyung_final_dataset.csv (raw counts, 51개) 

  ↓

① structure_risk 계산 (1st 모델 재실행)
   - 38개: 1_DatasetFor1stData에서 직접 매칭
   - 13개 미매칭: 광명시 평균값으로 대체 (imputation) 

  ↓     

② 피처 엔지니어링 (수동)
   - log1p 변환 (고왜도 컬럼)
   - interaction terms 5개
   - 최종 17개 피처 구성    

  ↓

③ 성남시 데이터로 학습한 모델로 예측 (2nd_model_calibrated.pkl.predict_proba(X_new))
   → risk_prob 산출     

  ↓     

④ safety_score = -log10(prob) min-max 정규화
   → qcut으로 A/B/C/D 등급    
   → gm_final.csv 저장    


In [37]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

print('라이브러리 로드 완료 ✅')

라이브러리 로드 완료 ✅


### 1. 광명시 데이터 구조 점수 산출하기

In [38]:
# 광명시 시설물 데이터 (raw count, 스케일 전)
gm_df = pd.read_csv('../preprocessing/data/5_gwangmyung_final_dataset.csv', encoding='utf-8-sig')
print(f'광명시 시설물: {len(gm_df)}개 스쿨존')
print(f'발생건수>0: {(gm_df["발생건수"]>0).sum()}개')
print(f'컬럼: {gm_df.columns.tolist()}')

# 구조 이미지 피처 데이터 (1차 모델 학습용)
first_df = pd.read_csv('./1_DatasetFor1stData.csv')
print(f'\n1_DatasetFor1stData: {len(first_df)}행')
print(f'광명시 이미지 데이터: {(first_df["시군구"]=="경기도 광명시").sum()}행')

광명시 시설물: 51개 스쿨존
발생건수>0: 4개
컬럼: ['시설물명', '위도', '경도', '도로안전표지', '과속방지턱', '도로적색표면', '무단횡단방지펜스', '무인교통단속카메라', '보호구역표지판', '생활안전CCTV', '신호등', '옐로카펫', '횡단보도', '보호구역 도로폭', '발생건수', '사상자수', '사망및중상자수', '사망자수', '중상자수', '경상자수', '동', '총인구수', '0~4세', '5~9세', '10~14세', '어린이 총인구', '어린이 비율(%)']

1_DatasetFor1stData: 3809행
광명시 이미지 데이터: 50행


### 1-2. 1차 모델 학습 및 구조 위험 점수 계산
`1_DatasetFor1stData.csv` 전체(전국)로 1차 모델을 학습한 뒤, 광명시 이미지 데이터에 적용해 `structure_risk`(사고 확률)를 산출합니다.

In [39]:
STRUCT_FEATURES = ['p_wide', 'p_barrier_yes', 'parked_density', 'sidewalk_ratio', 'road_width_relative']

# 전국 데이터로 1차 모델 학습
X_1st = first_df[STRUCT_FEATURES]
y_1st = first_df['accident_label']

model_1st = Pipeline([
    ('scaler', StandardScaler()),
    ('logit',  LogisticRegression(max_iter=1000))
])
model_1st.fit(X_1st, y_1st)
print(f'1차 모델 학습 완료 ✅  (전체 {len(first_df)}개 이미지)')

# 광명시 이미지만 필터링 → structure_risk 예측
gm_1st = first_df[first_df['시군구'] == '경기도 광명시'].copy()
gm_1st['structure_risk'] = model_1st.predict_proba(gm_1st[STRUCT_FEATURES])[:, 1]

print(f'\n광명시 이미지 {len(gm_1st)}개 → structure_risk 예측 완료')
print(gm_1st[['시설물명', 'structure_risk']].head(10).to_string(index=False))

1차 모델 학습 완료 ✅  (전체 3809개 이미지)

광명시 이미지 50개 → structure_risk 예측 완료
                   시설물명  structure_risk
                  가림유치원        0.230849
경기도 광명시 광명동(광명6동삼거리 부근)        0.154317
경기도 광명시 광명동(광문초교사거리 부근)        0.719220
  경기도 광명시 광명동(대성부동산 부근)        0.258493
   경기도 광명시 광명동(은성상사 부근)        0.420747
   경기도 광명시 광명동(해동검도 부근)        0.636229
  경기도 광명시 소하동(소하사거리 부근)        0.778706
   경기도 광명시 소하동(충현초교 부근)        0.635368
  경기도 광명시 철산동(광명북초교 부근)        0.316099
경기도 광명시 철산동(광북초교사거리 부근)        0.517608


### 1-3. 구조 점수 병합 및 미매칭 처리
- 시설물명 기준으로 이미지별 `structure_risk` 평균 → 시설물 단위로 집계
- 매칭 안 된 시설물(이미지 데이터 없음): 광명시 전체 평균으로 대체

In [40]:
# 시설물명별 평균 structure_risk
risk_per_facility = (
    gm_1st.groupby('시설물명')['structure_risk']
    .mean()
    .reset_index()
)

# 광명시 시설물 데이터와 병합
gm_df = gm_df.merge(risk_per_facility, on='시설물명', how='left')

# 미매칭 시설물 → 광명시 매칭 평균으로 대체
mean_risk   = gm_df['structure_risk'].mean()
n_unmatched = gm_df['structure_risk'].isna().sum()
unmatched_names = gm_df[gm_df['structure_risk'].isna()]['시설물명'].tolist()

gm_df['structure_risk'] = gm_df['structure_risk'].fillna(mean_risk)

print(f'매칭:              {len(gm_df) - n_unmatched}개')
print(f'미매칭 (평균 대체): {n_unmatched}개  →  평균값 {mean_risk:.4f}')
if unmatched_names:
    print(f'미매칭 시설물: {unmatched_names}')
print(f'\nstructure_risk 범위: {gm_df["structure_risk"].min():.4f} ~ {gm_df["structure_risk"].max():.4f}')
print(gm_df[['시설물명', 'structure_risk']].head())

매칭:              39개
미매칭 (평균 대체): 12개  →  평균값 0.3624
미매칭 시설물: ['예림유치원', '하안남초등학교', '예솔유치원', '가림초등학교', '홍익어린이집', '광덕초등학교', '예원유치원', '예지유치원', '예지유치원', '삼흥유치원', '혜성유치원', '철산초등학교']

structure_risk 범위: 0.0486 ~ 0.8039
        시설물명  structure_risk
0    빛가온초등학교        0.612305
1     빛가온유치원        0.567528
2  광명생명숲어린이집        0.499615
3     큰별어린이집        0.186229
4     충현초등학교        0.341508


### Step 2. 피처 엔지니어링
`2nd_Model_final.ipynb` 학습 시와 **동일한 변환**을 적용해야 모델 입력 형태가 일치합니다.
- `LOG_COLS`: 성남 학습 데이터 기준으로 왜도 > 1.0 이었던 컬럼 (하드코딩)
- 상호작용 항: `structure_risk × 핵심 시설물`

In [41]:
FACILITY_COLS = [
    '도로안전표지', '도로적색표면', '무단횡단방지펜스', '무인교통단속카메라',
    '보호구역표지판', '생활안전CCTV', '신호등', '옐로카펫', '횡단보도'
]

# 2nd_Model_final.ipynb 학습 시 성남 데이터 기준으로 결정된 log 변환 컬럼
# (재계산하면 광명 분포가 달라 불일치 발생 → 하드코딩)
LOG_COLS = ['무인교통단속카메라', '생활안전CCTV', '어린이 총인구']

for col in LOG_COLS:
    gm_df[f'log_{col}'] = np.log1p(gm_df[col])

# structure_risk × 핵심 시설물 상호작용 항
INTERACTION_COLS = ['생활안전CCTV', '무인교통단속카메라', '무단횡단방지펜스', '보호구역표지판', '도로적색표면']
for col in INTERACTION_COLS:
    gm_df[f'interact_{col}'] = gm_df['structure_risk'] * gm_df[col]

# 최종 17개 피처 (학습 시와 동일한 순서)
def fname(col):
    return f'log_{col}' if col in LOG_COLS else col

features_2nd = (
    ['structure_risk'] +
    [fname(c) for c in FACILITY_COLS] +
    [fname('어린이 총인구'), '어린이 비율(%)'] +
    [f'interact_{c}' for c in INTERACTION_COLS]
)

print(f'최종 피처: {len(features_2nd)}개')
for f in features_2nd:
    tag = '🆕' if ('log_' in f or 'interact_' in f) else '  '
    print(f'  {tag} {f}')

최종 피처: 17개
     structure_risk
     도로안전표지
     도로적색표면
     무단횡단방지펜스
  🆕 log_무인교통단속카메라
     보호구역표지판
  🆕 log_생활안전CCTV
     신호등
     옐로카펫
     횡단보도
  🆕 log_어린이 총인구
     어린이 비율(%)
  🆕 interact_생활안전CCTV
  🆕 interact_무인교통단속카메라
  🆕 interact_무단횡단방지펜스
  🆕 interact_보호구역표지판
  🆕 interact_도로적색표면


### Step 3. 추론 — `2nd_model_calibrated.pkl` 적용
기존 성남 학습 모델을 광명 데이터에 **전이(transfer)** 하여 사고 확률을 예측합니다.

In [42]:
calibrated = joblib.load('./2nd_model_calibrated.pkl')

X_gm = gm_df[features_2nd].fillna(gm_df[features_2nd].median())
gm_df['risk_prob'] = calibrated.predict_proba(X_gm)[:, 1]

print('추론 완료 ✅')
print(f'risk_prob 범위: {gm_df["risk_prob"].min():.4f} ~ {gm_df["risk_prob"].max():.4f}')
print()
print(gm_df[['시설물명', 'structure_risk', 'risk_prob']]
      .sort_values('risk_prob', ascending=False)
      .head(10)
      .to_string(index=False))

추론 완료 ✅
risk_prob 범위: 0.0037 ~ 0.0782

     시설물명  structure_risk  risk_prob
광명푸른숲어린이집        0.394421   0.078228
  광명남초등학교        0.242815   0.077545
 엄지창의어린이집        0.500381   0.059906
  광명동초등학교        0.148651   0.052819
   광일초등학교        0.048557   0.044852
   가림초등학교        0.362353   0.043017
   큰별어린이집        0.186229   0.041271
    아란유치원        0.150013   0.038158
    예솔유치원        0.362353   0.037694
   철산어린이집        0.462649   0.035591


### Step 4. 안전 점수 및 등급 산출
- `safety_score`: `-log10(risk_prob)` → min-max 정규화 (0~100, 클수록 안전)
- `safety_grade`: 4분위 기반 A(안전) / B / C / D(위험)

In [43]:
# -log10 변환 → min-max 정규화 (0~100)
log_risk = -np.log10(gm_df['risk_prob'].clip(lower=1e-6))
gm_df['safety_score'] = (
    (log_risk - log_risk.min()) / (log_risk.max() - log_risk.min()) * 100
)

# 4분위 등급 (A=안전, D=위험)
gm_df['safety_grade'] = pd.qcut(
    gm_df['safety_score'],
    q=4,
    labels=['D', 'C', 'B', 'A']
)

print('=== 안전등급 분포 ===')
for g in ['A', 'B', 'C', 'D']:
    cnt    = (gm_df['safety_grade'] == g).sum()
    scores = gm_df[gm_df['safety_grade'] == g]['safety_score']
    print(f'  {g}등급: {cnt}개 ({cnt/len(gm_df)*100:.1f}%)  '
          f'점수 범위: {scores.min():.1f} ~ {scores.max():.1f}')

# 결과 저장
out_cols = ['시설물명', '위도', '경도', '발생건수',
            'structure_risk', 'risk_prob', 'safety_score', 'safety_grade']
result_df = gm_df[out_cols].sort_values('risk_prob', ascending=False).reset_index(drop=True)

result_df.to_csv('./gm_final.csv', index=False, encoding='utf-8-sig')
print('\n저장 완료 ✅  →  gm_final.csv')
print()
print(result_df.to_string(index=False))

=== 안전등급 분포 ===
  A등급: 13개 (25.5%)  점수 범위: 59.5 ~ 100.0
  B등급: 12개 (23.5%)  점수 범위: 47.9 ~ 58.1
  C등급: 13개 (25.5%)  점수 범위: 31.5 ~ 47.0
  D등급: 13개 (25.5%)  점수 범위: 0.0 ~ 30.7

저장 완료 ✅  →  gm_final.csv

     시설물명        위도         경도  발생건수  structure_risk  risk_prob  safety_score safety_grade
광명푸른숲어린이집 37.477792 126.850856     0        0.394421   0.078228      0.000000            D
  광명남초등학교 37.476247 126.852962     0        0.242815   0.077545      0.287980            D
 엄지창의어린이집 37.475695 126.849686     0        0.500381   0.059906      8.763986            D
  광명동초등학교 37.483390 126.863980     0        0.148651   0.052819     12.899086            D
   광일초등학교 37.471017 126.847025     0        0.048557   0.044852     18.269069            D
   가림초등학교 37.458199 126.880148     0        0.362353   0.043017     19.641009            D
   큰별어린이집 37.436146 126.877327     0        0.186229   0.041271     21.001836            D
    아란유치원 37.437262 126.879689     0        0.150013   0.038158     23.57

### Step 5. 통합 CSV 생성
시설물 변수 + 구조 변수(이미지 피처 평균) + 파생 피처 + 점수/등급을 모두 포함한 통합 데이터를 저장합니다.

In [44]:
# 시설물명별 구조 변수 평균 (1차 모델 입력 피처)
STRUCT_RAW_COLS = ['p_wide', 'p_narrow', 'p_barrier_yes', 'p_barrier_no',
                   'road_width_relative', 'sidewalk_ratio', 'parked_density']

struct_per_facility = (
    gm_1st.groupby('시설물명')[STRUCT_RAW_COLS]
    .mean()
    .reset_index()
)

# gm_df에 구조 변수 병합
integrated = gm_df.merge(struct_per_facility, on='시설물명', how='left')

# 통합 컬럼 순서 정의
info_cols      = ['시설물명', '위도', '경도', '동']
accident_cols  = ['발생건수', '사상자수', '사망및중상자수', '사망자수', '중상자수', '경상자수']
facility_cols  = ['도로안전표지', '과속방지턱', '도로적색표면', '무단횡단방지펜스',
                  '무인교통단속카메라', '보호구역표지판', '생활안전CCTV',
                  '신호등', '옐로카펫', '횡단보도', '보호구역 도로폭']
pop_cols       = ['총인구수', '0~4세', '5~9세', '10~14세', '어린이 총인구', '어린이 비율(%)']
struct_cols    = STRUCT_RAW_COLS  # p_wide, p_narrow, ...
derived_cols   = ([f'log_{c}' for c in LOG_COLS] +
                  [f'interact_{c}' for c in INTERACTION_COLS])
score_cols     = ['structure_risk', 'risk_prob', 'safety_score', 'safety_grade']

# 실제 존재하는 컬럼만 선택
def select_existing(df, cols):
    return [c for c in cols if c in df.columns]

final_cols = (
    select_existing(integrated, info_cols) +
    select_existing(integrated, accident_cols) +
    select_existing(integrated, facility_cols) +
    select_existing(integrated, pop_cols) +
    select_existing(integrated, struct_cols) +
    select_existing(integrated, derived_cols) +
    select_existing(integrated, score_cols)
)

integrated_df = integrated[final_cols].sort_values('risk_prob', ascending=False).reset_index(drop=True)

integrated_df.to_csv('./gm_integrated.csv', index=False, encoding='utf-8-sig')
print(f'통합 CSV 저장 완료 ✅  →  gm_integrated.csv')
print(f'총 {len(integrated_df)}행 × {len(final_cols)}컬럼')
print(f'\n포함 컬럼 ({len(final_cols)}개):')
for i, c in enumerate(final_cols, 1):
    print(f'  {i:2d}. {c}')

통합 CSV 저장 완료 ✅  →  gm_integrated.csv
총 51행 × 46컬럼

포함 컬럼 (46개):
   1. 시설물명
   2. 위도
   3. 경도
   4. 동
   5. 발생건수
   6. 사상자수
   7. 사망및중상자수
   8. 사망자수
   9. 중상자수
  10. 경상자수
  11. 도로안전표지
  12. 과속방지턱
  13. 도로적색표면
  14. 무단횡단방지펜스
  15. 무인교통단속카메라
  16. 보호구역표지판
  17. 생활안전CCTV
  18. 신호등
  19. 옐로카펫
  20. 횡단보도
  21. 보호구역 도로폭
  22. 총인구수
  23. 0~4세
  24. 5~9세
  25. 10~14세
  26. 어린이 총인구
  27. 어린이 비율(%)
  28. p_wide
  29. p_narrow
  30. p_barrier_yes
  31. p_barrier_no
  32. road_width_relative
  33. sidewalk_ratio
  34. parked_density
  35. log_무인교통단속카메라
  36. log_생활안전CCTV
  37. log_어린이 총인구
  38. interact_생활안전CCTV
  39. interact_무인교통단속카메라
  40. interact_무단횡단방지펜스
  41. interact_보호구역표지판
  42. interact_도로적색표면
  43. structure_risk
  44. risk_prob
  45. safety_score
  46. safety_grade


### Step 6. 성남 결과 파일과 동일한 컬럼 구조로 광명 CSV 생성
- `3_final_gm.csv` : `3_final_scoring_results.csv` 컬럼 구조 동일
- `3_final_gm_improved.csv` : `3_final_scoring_results_improved.csv` 컬럼 구조 동일

In [45]:
# ─── 공통 준비 ────────────────────────────────────────────────────
# pipeline(SMOTE 포함) → 추론 시에는 scaler+logit만 동작
pipeline = joblib.load('./2nd_model_pipeline.pkl')
X_gm_inf = gm_df[features_2nd].fillna(gm_df[features_2nd].median())

gm_df['risk_prob_pipeline']    = pipeline.predict_proba(X_gm_inf)[:, 1]   # 원본용
gm_df['risk_prob_calibrated']  = gm_df['risk_prob']                        # improved용 (Step 3에서 저장)
gm_df['accident_label']        = (gm_df['발생건수'] >= 1).astype(int)

# ─── 3_final_gm.csv (원본 방식) ───────────────────────────────────
gm_df['risk_score_orig']    = gm_df['risk_prob_pipeline'] * 100
gm_df['safety_score_orig']  = (1 - gm_df['risk_prob_pipeline']) * 100
gm_df['safety_grade_orig']  = pd.qcut(
    gm_df['safety_score_orig'], q=4, labels=['D', 'C', 'B', 'A']
)

orig_col_map = {
    '시설물명': '시설물명', '위도': '위도', '경도': '경도',
    '도로안전표지': '도로안전표지', '도로적색표면': '도로적색표면',
    '무단횡단방지펜스': '무단횡단방지펜스', '무인교통단속카메라': '무인교통단속카메라',
    '보호구역표지판': '보호구역표지판', '생활안전CCTV': '생활안전CCTV',
    '신호등': '신호등', '옐로카펫': '옐로카펫', '횡단보도': '횡단보도',
    '동': '동', '발생건수': '발생건수', '사상자수': '사상자수',
    '사망및중상자수': '사망및중상자수', '사망자수': '사망자수',
    '중상자수': '중상자수', '경상자수': '경상자수',
    '총인구수': '총인구수', '0~4세': '0~4세', '5~9세': '5~9세',
    '10~14세': '10~14세', '어린이 총인구': '어린이 총인구', '어린이 비율(%)': '어린이 비율(%)',
    'p_wide': 'p_wide', 'p_narrow': 'p_narrow',
    'p_barrier_yes': 'p_barrier_yes', 'p_barrier_no': 'p_barrier_no',
    'road_width_relative': 'road_width_relative',
    'sidewalk_ratio': 'sidewalk_ratio', 'parked_density': 'parked_density',
    'accident_label': 'accident_label', 'structure_risk': 'structure_risk',
    'risk_prob_pipeline': 'risk_prob',       # 컬럼명을 원본과 동일하게
    'risk_score_orig': 'risk_score',
    'safety_score_orig': 'safety_score',
    'safety_grade_orig': 'safety_grade',
}

df_orig = (
    integrated                                   # 구조 변수(p_wide 등) 이미 병합된 df
    .assign(
        risk_prob_pipeline   = gm_df['risk_prob_pipeline'],
        risk_score_orig      = gm_df['risk_score_orig'],
        safety_score_orig    = gm_df['safety_score_orig'],
        safety_grade_orig    = gm_df['safety_grade_orig'],
        accident_label       = gm_df['accident_label'],
        structure_risk       = gm_df['structure_risk'],
    )
)

exist_orig_cols = [c for c in orig_col_map if c in df_orig.columns]
df_orig_out = df_orig[exist_orig_cols].rename(columns=orig_col_map)
df_orig_out = df_orig_out.sort_values('risk_prob', ascending=False).reset_index(drop=True)

df_orig_out.to_csv('./3_final_gm.csv', index=False, encoding='utf-8-sig')
print(f'3_final_gm.csv 저장 완료 ✅  ({len(df_orig_out)}행 × {len(df_orig_out.columns)}컬럼)')
print(df_orig_out[['시설물명', 'risk_prob', 'risk_score', 'safety_score', 'safety_grade']].head(5).to_string(index=False))

3_final_gm.csv 저장 완료 ✅  (51행 × 38컬럼)
     시설물명  risk_prob  risk_score  safety_score safety_grade
  광명남초등학교   0.340183   34.018320     65.981680            D
광명푸른숲어린이집   0.303270   30.326991     69.673009            D
 엄지창의어린이집   0.131693   13.169279     86.830721            D
  광명동초등학교   0.091169    9.116920     90.883080            D
   광일초등학교   0.049258    4.925815     95.074185            D


In [46]:
# ─── 3_final_gm_improved.csv (improved 방식) ─────────────────────
# safety_score / safety_grade 는 Step 4에서 이미 log 방식으로 계산됨

df_imp_out = gm_df[[
    '시설물명', '위도', '경도', '발생건수',
    'accident_label', 'structure_risk',
    'risk_prob_pipeline',   # → 컬럼명 risk_prob 으로 맞춤
    'risk_prob_calibrated', # → 컬럼명 risk_prob_calibrated
    'safety_score',         # log 방식
    'safety_grade',         # qcut A~D
]].rename(columns={'risk_prob_pipeline': 'risk_prob'}).copy()

df_imp_out = df_imp_out.sort_values('risk_prob_calibrated', ascending=False).reset_index(drop=True)

df_imp_out.to_csv('./3_final_gm_improved.csv', index=False, encoding='utf-8-sig')
print(f'3_final_gm_improved.csv 저장 완료 ✅  ({len(df_imp_out)}행 × {len(df_imp_out.columns)}컬럼)')
print(df_imp_out[['시설물명', 'risk_prob', 'risk_prob_calibrated', 'safety_score', 'safety_grade']].head(5).to_string(index=False))

3_final_gm_improved.csv 저장 완료 ✅  (51행 × 10컬럼)
     시설물명  risk_prob  risk_prob_calibrated  safety_score safety_grade
광명푸른숲어린이집   0.303270              0.078228      0.000000            D
  광명남초등학교   0.340183              0.077545      0.287980            D
 엄지창의어린이집   0.131693              0.059906      8.763986            D
  광명동초등학교   0.091169              0.052819     12.899086            D
   광일초등학교   0.049258              0.044852     18.269069            D
